In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
import re
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, coo_matrix

warnings.filterwarnings('ignore')

# --- THIS IS THE CORRECTED FUNCTION ---
# Custom SMAPE metric for LightGBM's .fit() method
def smape(y_true, y_pred):
    y_true_unlogged = np.expm1(y_true)
    y_pred_unlogged = np.expm1(y_pred)
    numerator = np.abs(y_pred_unlogged - y_true_unlogged)
    denominator = (np.abs(y_true_unlogged) + np.abs(y_pred_unlogged)) / 2
    smape_val = np.mean(numerator / (denominator + 1e-8)) * 100
    
    # This is the 3-item tuple that .fit() expects:
    # (name, value, is_higher_better)
    return 'smape', smape_val, False

# We no longer need make_scorer or GridSearchCV
print("Libraries imported and SMAPE function defined correctly.")

Libraries imported and SMAPE function defined correctly.


In [2]:
# --- Load Data ---
PROCESSED_DATA_FOLDER = '../data/processed'
TRAIN_FILE_PATH = os.path.join(PROCESSED_DATA_FOLDER, 'train_processed.parquet')
train_df = pd.read_parquet(TRAIN_FILE_PATH)
train_df['catalog_content'] = train_df['catalog_content'].astype(str)
print("Data loaded.")

# --- Extract Brand ---
brand_regex = re.compile(r'Item Name:\s*([\w\’\'\-\.&]+)')
def extract_brand(text_series):
    return text_series.str.extract(brand_regex, expand=False).fillna('Unknown').str.lower()

train_df['brand'] = extract_brand(train_df['catalog_content'])
brand_encoder = LabelEncoder()
train_df['brand_encoded'] = brand_encoder.fit_transform(train_df['brand'])
print("Brand feature created.")

# --- Get All Features ---
# This is our champion feature set from notebook 05
X_train, X_val = train_test_split(train_df, test_size=0.2, random_state=42)
y_train = X_train['log_price']
y_val = X_val['log_price']

# --- TF-IDF ---
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, stop_words='english', dtype=np.float32)
X_train_text = tfidf_vec.fit_transform(X_train['catalog_content'])
X_val_text = tfidf_vec.transform(X_val['catalog_content'])

# --- Combine Features ---
numerical_features = [col for col in train_df.columns if col.startswith('unit_') or col in ['pack_size', 'total_measure']]
X_train_num = X_train[numerical_features].values.astype(np.float32)
X_val_num = X_val[numerical_features].values.astype(np.float32)
X_train_brand = X_train[['brand_encoded']].values.astype(np.float32)
X_val_brand = X_val[['brand_encoded']].values.astype(np.float32)

X_train_final = hstack((X_train_text, coo_matrix(X_train_num), coo_matrix(X_train_brand)))
X_val_final = hstack((X_val_text, coo_matrix(X_val_num), coo_matrix(X_val_brand)))

print(f"Feature engineering complete. Training shape: {X_train_final.shape}")

Data loaded.
Brand feature created.
Feature engineering complete. Training shape: (60000, 20045)


In [3]:
print("Starting FAST Manual Hyperparameter Tuning...")

# We will test a few different parameter sets.
# The 'n_estimators' is high, but 'early_stopping' will find the
# optimal number automatically.

params_1 = {
    'learning_rate': 0.05,
    'n_estimators': 3000,
    'num_leaves': 31,  # Default
    'random_state': 42,
    'n_jobs': -1,
    'metric': 'None'
}

params_2 = {
    'learning_rate': 0.02,  # Slower learning
    'n_estimators': 5000,  # More trees
    'num_leaves': 40,
    'random_state': 42,
    'n_jobs': -1,
    'metric': 'None'
}

params_3 = {
    'learning_rate': 0.1,  # Faster learning
    'n_estimators': 2000,
    'num_leaves': 50,  # More complex
    'random_state': 42,
    'n_jobs': -1,
    'metric': 'None'
}

param_sets = [params_1, params_2, params_3]
best_overall_score = 999
best_overall_params = None

# Get the categorical feature index
categorical_feature_index = X_train_final.shape[1] - 1

# --- Loop through and test each set ---
for i, params in enumerate(param_sets):
    print(f"\n--- Testing Parameter Set {i+1} ---")
    print(params)
    
    lgbm_model = lgb.LGBMRegressor(**params)
    
    lgbm_model.fit(
        X_train_final, 
        y_train,
        eval_set=[(X_val_final, y_val)],
        eval_metric=smape,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100), # This is the crucial part
            lgb.log_evaluation(period=500)
        ]
    )
    
    best_score = lgbm_model.best_score_['valid_0']['smape']
    print(f"Result for Set {i+1}: SMAPE = {best_score:.4f}")
    
    if best_score < best_overall_score:
        best_overall_score = best_score
        best_overall_params = params

print("\n--- Manual Tuning Complete ---")
print(f"Best Overall SMAPE: {best_overall_score:.4f}")
print("Best Parameter Set:")
print(best_overall_params)

Starting FAST Manual Hyperparameter Tuning...

--- Testing Parameter Set 1 ---
{'learning_rate': 0.05, 'n_estimators': 3000, 'num_leaves': 31, 'random_state': 42, 'n_jobs': -1, 'metric': 'None'}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 3.098722 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1094295
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 19578
[LightGBM] [Info] Start training from score 2.740904
Training until validation scores don't improve for 100 rounds
[500]	valid_0's smape: 53.4912
[1000]	valid_0's smape: 52.3256
[1500]	valid_0's smape: 51.7794
[2000]	valid_0's smape: 51.5035
[2500]	valid_0's smape: 51.3105
[3000]	valid_0's smape: 51.2323
Did not meet early stopping. Best iteration is:
[2998]	valid_0's smape: 51.231
Result for Set 1: SMAPE = 51.2310

--- Testing Parameter Set 2 ---
{'learning_rate': 0.02, 'n_estimators': 5000, 'num_leaves'